# 🚀 NairaLLM — Google Colab GPU Training Workflow

**Objective**: End-to-end reproducible training workflow for NairaLLM using GitHub as the canonical source-of-truth and Google Colab Free Tesla T4 GPU.

### Architecture & Guarantees:
- **Canonical Source of Truth**: GitHub (`https://github.com/OMBOS921/my-naira-ai-assistant-.git`)
- **Compute Engine**: Google Colab Free Tier (Tesla T4 GPU, ~15 GB VRAM, Mixed Precision)
- **Zero Cost Policy**: `USE_PAID_COMPUTE = False` (Strictly zero paid compute units / no Colab Pro required)
- **Persistent Checkpoints**: Google Drive (`/content/drive/MyDrive/Naira-Training/checkpoints/`)
- **Strict Source Control**: Git commit SHA, branch, and Dataset SHA-256 recorded in every checkpoint
- **Safety & Gates**: GPU Smoke Test + Pilot Training Gate + Explicit User Activation for Full Training.

## 🛠️ Step 1: Environment & Free GPU Hardware Gate

Inspects runtime environment, checks CUDA GPU availability, and enforces free cloud resource policy.

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT & FREE GPU CHECK
# ==============================================================================
import os
import sys
import platform
import shutil

USE_PAID_COMPUTE = False

print("=" * 60)
print("     NAIRALLM — HARDWARE & RUNTIME DIAGNOSTIC CHECK      ")
print("=" * 60)
print(f"Python Version:   {platform.python_version()} ({platform.platform()})")
print(f"Policy:           USE_PAID_COMPUTE = {USE_PAID_COMPUTE} (Free Cloud GPU Only)")

try:
    import torch
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
    cuda_version = getattr(torch.version, "cuda", "N/A")
except ImportError:
    torch = None
    torch_version = "Not Installed"
    cuda_available = False
    cuda_version = "N/A"

print(f"PyTorch Version:  {torch_version}")
print(f"CUDA Available:   {cuda_available} (CUDA {cuda_version})")

if not cuda_available:
    print("\n" + "!" * 60)
    print("[STOP] NO CUDA GPU DETECTED!")
    print("Please enable a free GPU in Google Colab:")
    print("  1. Go to Menu: Runtime -> Change runtime type")
    print("  2. Hardware accelerator: Select 'T4 GPU'")
    print("  3. Click 'Save' and re-run this cell.")
    print("Execution halted to prevent slow/unsupported CPU training.")
    print("!" * 60 + "\n")
    raise RuntimeError("CUDA GPU is required for NairaLLM training. Halting execution.")

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
vram_total_gb = round(gpu_props.total_memory / (1024 ** 3), 2)

print(f"GPU Runtime:      {gpu_name}")
print(f"VRAM Total:       {vram_total_gb} GB")
print(f"Compute Cap:      {gpu_props.major}.{gpu_props.minor}")
print(f"Provider:         Google Colab (Free Tier T4)")
print("=" * 60)
print("[STATUS] FREE GPU HARDWARE GATE: PASSED")

## 📂 Step 2: Mount Google Drive for Persistent Storage

Mounts Google Drive so all training checkpoints, optimizer states, and evaluation reports persist across runtime restarts.

In [ ]:
# ==============================================================================
# 2. GOOGLE DRIVE PERSISTENT STORAGE MOUNT
# ==============================================================================
import os
from google.colab import drive

print("[STORAGE] Mounting Google Drive to persist checkpoints across sessions...")
drive.mount('/content/drive')

CHECKPOINT_ROOT = "/content/drive/MyDrive/Naira-Training/checkpoints"
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

print(f"\n[OK] Persistent Checkpoint Directory Ready at:\n     {CHECKPOINT_ROOT}")

## 🐙 Step 3: Repository Sync (GitHub Source-of-Truth)

Synchronizes source code directly from GitHub.
- **Public Repository**: Automatic `git clone` / `git pull`.
- **Private Repository**: Interactive secure token input (using masked `getpass`, never logged or saved to disk).

In [ ]:
# ==============================================================================
# 3. GITHUB REPOSITORY SYNC (CANONICAL SOURCE OF TRUTH)
# ==============================================================================
import os
import sys
from pathlib import Path
import getpass
import subprocess

REPO_URL = "https://github.com/OMBOS921/my-naira-ai-assistant-.git"
REPO_DIR = "/content/naira-os"
BRANCH = "main"

os.chdir("/content")

if os.path.exists(REPO_DIR):
    print(f"[REPO] Repository already exists at {REPO_DIR}. Pulling latest changes...")
    os.chdir(REPO_DIR)
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    print(f"[REPO] Cloning canonical repository: {REPO_URL}...")
    res = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], capture_output=True, text=True)
    if res.returncode == 0:
        print("[REPO] Clone successful!")
        os.chdir(REPO_DIR)
    else:
        print("[NOTICE] Standard clone failed. If repository is private, please authenticate.")
        print("Please provide a GitHub Personal Access Token (PAT) with read permissions.")
        print("(Input is masked with getpass and will NOT be saved or logged.)")
        token = getpass.getpass("Enter GitHub PAT (leave blank to skip): ").strip()
        if token:
            auth_url = REPO_URL.replace("https://", f"https://{token}@")
            clone_res = subprocess.run(["git", "clone", "-b", BRANCH, auth_url, REPO_DIR], capture_output=True, text=True)
            del token
            del auth_url
            if clone_res.returncode == 0:
                print("[REPO] Authenticated clone successful!")
                os.chdir(REPO_DIR)
                subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)
            else:
                print(f"[ERROR] Authenticated clone failed: {clone_res.stderr}")
                raise RuntimeError("Could not clone repository.")
        else:
            raise RuntimeError(f"Clone failed: {res.stderr}")

workspace_root = Path(REPO_DIR).resolve()
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print(f"\n[OK] Repository active at: {workspace_root}")
!git log -n 1 --oneline

## 📦 Step 4: Install Dependencies & Verify Source Control Provenance

Installs only required dependencies and verifies:
- Git commit SHA & branch
- Dataset A SHA-256 & record count
- Tokenizer vocabulary size
- Model configuration

In [ ]:
# ==============================================================================
# 4. DEPENDENCIES & SOURCE CONTROL VERIFICATION
# ==============================================================================
!pip install -q tokenizers psutil

import hashlib
import json
import subprocess
from pathlib import Path
from NairaLLM.model.tokenizer.naira_tokenizer import NairaTokenizer
from NairaLLM.model.config.model_config import NairaModelConfig

# 1. Git Provenance
git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
git_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()

# 2. Dataset A Provenance
ds_path = Path("NairaLLM/dataset/semantic_corpus/semantic_pretrain_v1_5_final.jsonl")
if not ds_path.exists():
    ds_path = Path("NairaLLM/dataset/semantic_corpus/semantic_pretrain_v1_5.jsonl")

h = hashlib.sha256()
with open(ds_path, "rb") as f:
    while chunk := f.read(65536):
        h.update(chunk)
ds_sha256 = h.hexdigest()

# 3. Tokenizer & Dataset Statistics
tok_path = Path("NairaLLM/model/tokenizer/naira_tokenizer.json")
tokenizer = NairaTokenizer(tok_path)

with open(ds_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

total_tokens = sum(len(tokenizer.encode(r.get("text", ""))) for r in records)

print("=" * 60)
print("        NAIRALLM TRAINING SOURCE CONTROL AUDIT          ")
print("=" * 60)
print(f"Git Commit SHA:       {git_sha}")
print(f"Git Branch:           {git_branch}")
print(f"Dataset A File:       {ds_path.name}")
print(f"Dataset A SHA-256:    {ds_sha256}")
print(f"Dataset A Records:    {len(records)} records ({total_tokens:,} tokens)")
print(f"Tokenizer Vocab:      {tokenizer.vocab_size} tokens ({tok_path.name})")
print("=" * 60)

## 🧪 Step 5: GPU Smoke Test (10-Point Pipeline Verification)

Runs the 10-point verification test before training:
1. Tokenizer validation
2. Model creation
3. Batch loading
4. Forward pass
5. Loss calculation
6. Backward pass (non-NaN gradients)
7. Optimizer step
8. Checkpoint serialization
9. Checkpoint reload parity
10. Resumed training step

In [ ]:
# ==============================================================================
# 5. RUN 10-STEP GPU SMOKE TEST
# ==============================================================================
from NairaLLM.training.cloud.run_smoke_test import run_gpu_smoke_test

smoke_results = run_gpu_smoke_test()
assert smoke_results.get("all_passed"), "Smoke test failed! Halting pipeline."
print("\n[SMOKE TEST] All 10 GPU pipeline checks PASSED successfully.")

## ⚡ Step 6: Pilot Training Run (Short Semantic Pilot + STOP Gate)

Executes a short pilot run (10 epochs) on Dataset A with AMP mixed precision.
- Checkpoints saved to Google Drive: `/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretrain_pilot/`
- Evaluates semantic foundations across 7 domains
- Halts at the STOP Gate with clear recommendation.

In [ ]:
# ==============================================================================
# 6. RUN SEMANTIC PRETRAINING PILOT (SHORT PILOT + STOP GATE)
# ==============================================================================
from NairaLLM.training.scripts.run_semantic_pilot import run_semantic_pilot

pilot_checkpoint_dir = "/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretrain_pilot"
os.makedirs(pilot_checkpoint_dir, exist_ok=True)

pilot_results = run_semantic_pilot(
    epochs=10,
    batch_size=4,
    grad_accum_steps=4,
    learning_rate=4e-4,
    max_seq_len=256,
    custom_checkpoint_dir=pilot_checkpoint_dir,
)

print(f"\n[PILOT RESULT] Recommendation: {pilot_results.get('recommendation')}")
print(f"[PILOT RESULT] Rationale:      {pilot_results.get('recommendation_reason')}")
print("\n[SAVED CHECKPOINTS IN GOOGLE DRIVE]:")
!ls -lh "{pilot_checkpoint_dir}"

## 🏁 Step 7: Full Training Run (Explicit User Action Required)

> [!IMPORTANT]
> Full training requires an **explicit user action** to initiate.
> Set `RUN_FULL_TRAINING = True` below when ready to launch the complete training run.

In [ ]:
# ==============================================================================
# 7. FULL TRAINING RUN (EXPLICIT USER INVOCATION ONLY)
# ==============================================================================
RUN_FULL_TRAINING = False  # Set to True to initiate full training

RESUME_FROM_CHECKPOINT = True
FULL_TRAIN_EPOCHS = 25
FULL_BATCH_SIZE = 4
FULL_GRAD_ACCUM = 4
FULL_LEARNING_RATE = 4e-4

full_checkpoint_dir = "/content/drive/MyDrive/Naira-Training/checkpoints/full_pretraining"
os.makedirs(full_checkpoint_dir, exist_ok=True)

if not RUN_FULL_TRAINING:
    print("=" * 60)
    print("[POLICY] FULL TRAINING IS CURRENTLY PAUSED")
    print("To initiate the full training run:")
    print("  1. Change RUN_FULL_TRAINING = True in this cell.")
    print("  2. Re-run this cell.")
    print("=" * 60)
else:
    print("=" * 60)
    print(f"[LAUNCH] Starting Full Training Run ({FULL_TRAIN_EPOCHS} Epochs)...")
    print("=" * 60)
    from NairaLLM.training.scripts.train_gpu import train_gpu

    train_metadata = train_gpu(
        dataset_path=str(ds_path),
        checkpoint_dir=full_checkpoint_dir,
        epochs=FULL_TRAIN_EPOCHS,
        batch_size=FULL_BATCH_SIZE,
        grad_accum_steps=FULL_GRAD_ACCUM,
        learning_rate=FULL_LEARNING_RATE,
        max_seq_len=256,
        resume=RESUME_FROM_CHECKPOINT,
        require_free_gpu=True,
    )

    print("\n[COMPLETED] Full training run completed successfully!")
    print("\nCheckpoints saved to Google Drive:")
    !ls -lh "{full_checkpoint_dir}"